# 05 — Project Phase 4: State-Reconstruction Feature Assembly & Proof

**Project Phase 4** (`docs/PROJECT_PLAN.md`, `docs/PHASE4_EXECUTION_PLAN.md`) built the state-reconstruction
and feature-engineering layer steps 4.1-4.8: `StateSnapshot` (4.1), shrinkage-regularized location
signatures (4.2), spatial-neighbor history (4.3), the `Transformer` protocol + config registry (4.4),
temporal/seasonal features (4.5), environmental features (4.6), five target-transformation framings (4.7),
and a per-module leakage-firewall proof (4.8).

This notebook is step **4.9**: compose everything into one flat feature matrix
(`features.assemble.build_feature_matrix`), prove the *composed whole* has no leakage (not just each module
in isolation), settle the target-transformation question (4.7) with real numbers, directly confirm A-014
now that the shrinkage-regularized signature this finding motivated actually exists, and run a real
feature-importance pass decomposed by regime -- this phase's actual Definition of Done requirement.

**Run locally, not in the cloud session** -- same wall-clock-budget reasoning notebooks 03/04 both
documented: a real LightGBM fit against the full ~2.15M-row `Train.csv`, repeated once per target transform
plus the full feature-importance pass, is a multi-minute-per-cell workload better run against the real conda
environment (`tws-forecast`) than a metered cloud session. Requires `lightgbm` in that environment
(`pip install lightgbm` if not already present -- every other import here is already a project dependency).

## 1. Setup

Loads the real `Train.csv` and the Phase 1 constants every later number in this notebook is checked against.

In [ ]:
import gc
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.inspection import permutation_importance

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from tws_forecast.data.loaders import load_train
from tws_forecast.state.reconstruction import location_id_from_lat_lon
from tws_forecast.utils.seeds import RANDOM_SEED, set_seed
from tws_forecast.validation.phase1_constants import (
    BASELINE_A, BASELINE_B, BASELINE_C, BASELINE_D, PROMOTION_THRESHOLDS,
)
from tws_forecast.validation.decomposition import decompose, degradation_slope, ACF_QUARTILE_ORDER
from tws_forecast.validation.splitters import expanding_window_splits
from tws_forecast.validation.harness import CandidateReport, evaluate_candidate, promote
from tws_forecast.validation.experiment_log import log_candidate
from tws_forecast.validation.leakage_tests import future_row_shuffle_test

from tws_forecast.models.baselines import SeasonalClimatologyPredictor
from tws_forecast.features.assemble import build_feature_matrix
from tws_forecast.state.signatures import LocationSignatureTransformer
from tws_forecast.features.targets import TARGET_TRANSFORMS, LevelTargetTransform

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)
set_seed(RANDOM_SEED)

def savefig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved {path}")

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


In [ ]:
t0 = time.time()
train = load_train()
print(f"Loaded Train.csv in {time.time()-t0:.1f}s: {train.shape[0]:,} rows, {train.shape[1]} columns, "
      f"{train['time'].min().date()} to {train['time'].max().date()}")
print("Passed pandera schema validation (including the full-grid check) at load time.")


## 2. Per-location ACF(1) lookup

Reused from `notebooks/03_validation_harness.ipynb` §3 / `notebooks/04_baselines.ipynb` §2 -- the same real,
per-location ACF(1) computation, feeding `decompose()`'s `staleness_x_acf_quartile` cross-cut and
`degradation_slope()`'s AR(1) reference overlay for every candidate below. Kept notebook-local rather than
promoted to a shared module, per the same reasoning notebook 04 documented there.

In [ ]:
t0 = time.time()
full_sorted = train.sort_values(["lat", "lon", "time"])
full_sorted = full_sorted.assign(TWS_prev=full_sorted.groupby(["lat", "lon"])["TWS_t"].shift(1))
acf1_df = (
    full_sorted.dropna(subset=["TWS_prev"])
    .groupby(["lat", "lon"])
    .apply(lambda g: g["TWS_t"].corr(g["TWS_prev"]))
    .rename("acf1")
    .reset_index()
)
acf1_df["location_id"] = [
    location_id_from_lat_lon(lat, lon) for lat, lon in zip(acf1_df["lat"], acf1_df["lon"])
]
acf_lookup = acf1_df.set_index("location_id")["acf1"]
print(f"Computed per-location ACF(1) for {len(acf_lookup):,} locations in {time.time()-t0:.1f}s")
del full_sorted
gc.collect()


## 3. Build the full feature matrix across a few CV folds

Sanity-checks `features.assemble.build_feature_matrix()` end-to-end against the real, full-scale grid
(~15,715 locations) before any model touches it -- shapes, timing, and column composition, matching this
project's now-established "prove the plumbing before scoring anything" pattern.

In [ ]:
t0 = time.time()
demo_folds = list(expanding_window_splits(train, n_folds=3, val_window_months=6))
for i, (fold_train, fold_val) in enumerate(demo_folds):
    ft0 = time.time()
    matrix = build_feature_matrix(fold_val, train_df=fold_train)
    print(
        f"fold {i}: train={len(fold_train):,} rows, val={len(fold_val):,} rows, "
        f"matrix={matrix.shape[0]:,}x{matrix.shape[1]} columns, built in {time.time()-ft0:.1f}s"
    )
    if i == 0:
        print("\ncolumns:", matrix.columns.tolist())
        display(matrix.head(3))
    del matrix
    gc.collect()
print(f"\nTotal: {time.time()-t0:.1f}s across {len(demo_folds)} folds")


## 4. Leakage proof: future-row shuffle test on the assembled pipeline

Step 4.8 already proved every individual `Transformer` has no leakage in isolation
(`tests/test_no_leakage_features.py`). This is the composed-whole version, run literally against the real
data: fit a real LightGBM model on `build_feature_matrix()`'s output, then confirm shuffling every row's
*position* after a cutoff never changes a prediction made at or before it.

In [ ]:
_STATE_STATUS_CATEGORIES = ["OBSERVED", "RECONSTRUCTED", "PARTIALLY_RECONSTRUCTED"]


def _encode_categoricals(matrix: pd.DataFrame) -> pd.DataFrame:
    """LightGBM's sklearn API natively handles pandas 'category' dtype
    columns -- state_state_status (the only non-numeric column
    build_feature_matrix produces, see tests/test_assemble.py) is cast to a
    *fixed* category set (not inferred per-call) so fit and predict always
    agree on category codes even if one particular frame happens not to
    contain every status."""
    matrix = matrix.copy()
    if "state_state_status" in matrix.columns:
        matrix["state_state_status"] = pd.Categorical(
            matrix["state_state_status"], categories=_STATE_STATUS_CATEGORIES
        )
    return matrix


class LightGBMFeaturePredictor:
    """validation.tiers.Predictor wrapping build_feature_matrix() + a
    LightGBM regressor, optionally trained in a transformed target space
    via a features.targets.TargetTransform (default: LevelTargetTransform,
    i.e. no transformation) -- the vehicle for both section 5's
    target-transformation comparison and section 7's feature-importance
    pass."""

    def __init__(self, target_transform=None, lgb_params=None, assemble_kwargs=None):
        self.target_transform = target_transform or LevelTargetTransform()
        self.lgb_params = {
            "objective": "regression",
            "n_estimators": 400,
            "learning_rate": 0.05,
            "num_leaves": 63,
            "min_child_samples": 50,
            "random_state": RANDOM_SEED,
            "verbosity": -1,
        }
        if lgb_params:
            self.lgb_params.update(lgb_params)
        self.assemble_kwargs = assemble_kwargs or {}
        self._train_df = None
        self._model = None
        self._feature_columns = None

    def fit(self, train_df: pd.DataFrame) -> None:
        self._train_df = train_df.copy()
        matrix = _encode_categoricals(build_feature_matrix(train_df, train_df=train_df, **self.assemble_kwargs))
        self._feature_columns = matrix.columns.tolist()
        y = self.target_transform.forward(train_df).to_numpy()
        self._model = lgb.LGBMRegressor(**self.lgb_params)
        self._model.fit(matrix, y, categorical_feature=[
            c for c in matrix.columns if str(matrix[c].dtype) == "category"
        ])

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        matrix = _encode_categoricals(build_feature_matrix(df, train_df=self._train_df, **self.assemble_kwargs))
        matrix = matrix.reindex(columns=self._feature_columns, fill_value=0.0)
        preds_transformed = self._model.predict(matrix)
        return self.target_transform.inverse(preds_transformed, df).to_numpy()


In [ ]:
t0 = time.time()
shuffle_model = LightGBMFeaturePredictor()
leakage_ok = future_row_shuffle_test(shuffle_model, train, cutoff_time="2010-12-01")
print(f"future_row_shuffle_test on the assembled pipeline: {'PASSED' if leakage_ok else 'FAILED'} "
      f"({time.time()-t0:.1f}s)")
assert leakage_ok, "Leakage detected in the assembled feature-matrix pipeline -- stop and investigate before proceeding."


## 5. Target-transformation comparison (step 4.7)

Per `docs/ARCHITECTURE.md` §9, this is "a first-class experimental question," not a foregone conclusion.
One LightGBM trained per transform (`features.targets.TARGET_TRANSFORMS`), scored through Tier 1 + Tier 2
with identical folds and identical features otherwise -- only the target framing changes. The winner becomes
this project's default target framing from here on; if it isn't `"level"`, that's recorded as a short ADR
(`docs/adr/0007-*.md`, drafted after this cell actually runs and a real winner is known).

Run with `evaluate_candidate`'s `tier1_scenario="expanding_window_quick"`/`tier2_scenario="blackout_curve_quick"` overrides -- this is an exploratory/comparative pass across 5 candidates, not a report a `promote()` decision is based on, and the full 5-fold/15-window scenarios here would multiply an already multi-minute-per-fit cost by 5 candidates for no decision-relevant precision gain. Section 7's real Definition-of-Done proof stays on the full-rigor `expanding_window`/`blackout_curve` scenarios.


In [ ]:
target_transform_reports: dict[str, CandidateReport] = {}

for name, transform in TARGET_TRANSFORMS.items():
    t0 = time.time()
    model = LightGBMFeaturePredictor(target_transform=transform)
    report = evaluate_candidate(
        model, train, candidate_id=f"target_transform_{name}",
        acf_lookup=acf_lookup, include_tier3=False,
        tier1_scenario="expanding_window_quick", tier2_scenario="blackout_curve_quick",
    )
    target_transform_reports[name] = report
    print(f"[{name}] tier1={report.tier1.overall_rmse:.4f}  tier2={report.tier2.overall_rmse:.4f}  "
          f"({time.time()-t0:.1f}s)")


In [ ]:
target_transform_summary = pd.DataFrame(
    [
        {"transform": name, "tier1_rmse": r.tier1.overall_rmse, "tier2_rmse": r.tier2.overall_rmse}
        for name, r in target_transform_reports.items()
    ]
).sort_values("tier2_rmse").reset_index(drop=True)
print(target_transform_summary.to_string(index=False))

winning_transform = target_transform_summary.iloc[0]["transform"]
print(f"\nWinner (lowest Tier 2 RMSE): {winning_transform!r}")
if winning_transform != "level":
    print(
        "ACTION REQUIRED: the winner is not 'level' -- draft docs/adr/0007-target-transformation-default.md "
        "recording this decision per docs/ARCHITECTURE.md section 2's ADR mechanism, and adopt "
        f"{winning_transform!r} as this project's default target framing from here on."
    )
else:
    print("'level' wins -- no default change, no new ADR needed.")


## 6. A-014 direct confirmation

`docs/ASSUMPTIONS.md` A-014: Project Phase 3's `SeasonalClimatologyPredictor` (Baseline C, naive
per-`(location, calendar-month)` climatology) scored **1.0796** out-of-fold -- worse than
`GlobalMeanPredictor`'s **0.8740** -- confirming naive climatology overfits and motivating step 4.2's
shrinkage-regularized location signatures. This section is the "direct confirmation still pending" half of
that finding: does the *shrinkage-regularized* signature actually beat 1.0796, and ideally 0.8740 too, on
the identical folds?

Also run with the quick scenarios (`tier1_scenario="expanding_window_quick"`/`tier2_scenario="blackout_curve_quick"`) -- this section compares two simple mean-only predictors against each other and against Phase 3's already-recorded numbers; it doesn't need full-rigor folds to answer "does shrinkage beat naive climatology" directionally.


In [ ]:
class SignatureMeanPredictor:
    """validation.tiers.Predictor stand-in for step 4.2's shrinkage-regularized
    signature: predicts each row's origin-time-indexed, shrinkage-regularized
    location mean directly (state.signatures.LocationSignatureTransformer),
    with no other feature or model on top -- the most direct possible
    apples-to-apples comparison against Baseline C, which is likewise "just
    a location-level mean," but naive/unshrunk."""

    def __init__(self, shrinkage_k=None):
        self._shrinkage_k = shrinkage_k
        self._transformer = None

    def fit(self, train_df: pd.DataFrame) -> None:
        self._transformer = LocationSignatureTransformer(shrinkage_k=self._shrinkage_k)
        self._transformer.fit(train_df)

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        signatures = self._transformer.transform(df)
        return signatures["mean"].to_numpy(dtype=float)


t0 = time.time()
signature_model = SignatureMeanPredictor()
signature_report = evaluate_candidate(
    signature_model, train, candidate_id="a014_shrinkage_signature_mean",
    acf_lookup=acf_lookup, include_tier3=False,
    tier1_scenario="expanding_window_quick", tier2_scenario="blackout_curve_quick",
)
print(f"Shrinkage-regularized signature mean: tier1={signature_report.tier1.overall_rmse:.4f}  "
      f"tier2={signature_report.tier2.overall_rmse:.4f}  ({time.time()-t0:.1f}s)")

t0 = time.time()
climatology_model = SeasonalClimatologyPredictor()
climatology_report = evaluate_candidate(
    climatology_model, train, candidate_id="a014_naive_climatology_rerun",
    acf_lookup=acf_lookup, include_tier3=False,
    tier1_scenario="expanding_window_quick", tier2_scenario="blackout_curve_quick",
)
print(f"Naive climatology (Baseline C, rerun on identical folds): tier1={climatology_report.tier1.overall_rmse:.4f}  "
      f"tier2={climatology_report.tier2.overall_rmse:.4f}  ({time.time()-t0:.1f}s)")

print(f"\nA-014 reference: naive climatology = 1.0796 (Phase 3), global mean = 0.8740 (Phase 3)")
print(f"Shrinkage signature beats naive climatology (1.0796): "
      f"{signature_report.tier2.overall_rmse < 1.0796}")
print(f"Shrinkage signature beats global mean (0.8740): "
      f"{signature_report.tier2.overall_rmse < 0.8740}")


## 7. Feature-importance pass, decomposed by regime

This phase's actual Definition of Done requirement, not a promotion decision (Project Phase 5 owns
promotion, against the full champion ladder). A LightGBM trained on the *full* assembled feature matrix
(raw columns + `StateSnapshot` fields + signatures + spatial-history + temporal + environmental), run
through `harness.evaluate_candidate()` exactly like Project Phase 3's six baselines were, using the winning
target transform from section 5. Native (gain-based) importance is global; permutation importance is
additionally computed *separately* on the masked-regime and observed-regime subsets of one held-out fold --
the "decomposed by regime" half of the Definition of Done -- checked against A-008/A-010's prediction that
`acf_1_3_6_12`/`months_since_observation`-family features should dominate specifically in the masked
regime.

In [ ]:
winning_transform_obj = TARGET_TRANSFORMS[winning_transform]

t0 = time.time()
importance_model = LightGBMFeaturePredictor(target_transform=winning_transform_obj)
importance_report = evaluate_candidate(
    importance_model, train, candidate_id="phase4_full_feature_matrix",
    acf_lookup=acf_lookup, include_tier3=True, n_anchors=3,
)
print(f"Full feature matrix ({winning_transform!r} target): tier1={importance_report.tier1.overall_rmse:.4f}  "
      f"tier2={importance_report.tier2.overall_rmse:.4f}  "
      f"tier3={importance_report.tier3.overall_rmse if importance_report.tier3 else float('nan'):.4f}  "
      f"({time.time()-t0:.1f}s)")

print(f"\nBaseline D floor (Tier 2, realistic no-ML floor): {BASELINE_D}")
print(f"Beats Baseline D: {importance_report.tier2.overall_rmse < BASELINE_D}")

hard_bucket_rows = importance_report.tier2_decomposition[
    importance_report.tier2_decomposition["slice_value"].isin(["k=5", "k=6", "k=7"])
]
print("\nHardest staleness buckets (k=5, 6, 7):")
print(hard_bucket_rows.to_string(index=False))


In [ ]:
gain_importance = pd.Series(
    importance_model._model.booster_.feature_importance(importance_type="gain"),
    index=importance_model._feature_columns,
).sort_values(ascending=False)
print("Top 25 features by native (gain) importance, full model:")
print(gain_importance.head(25).to_string())

fig, ax = plt.subplots(figsize=(8, 8))
gain_importance.head(25).sort_values().plot.barh(ax=ax, color="#2c7fb8")
ax.set_xlabel("Gain importance")
ax.set_title("Top 25 features -- full assembled matrix, gain importance")
savefig(fig, "05_gain_importance_top25.png")
plt.show()


In [ ]:
t0 = time.time()
_, perm_val_fold = next(expanding_window_splits(train, n_folds=1, val_window_months=6))
perm_matrix = _encode_categoricals(
    build_feature_matrix(perm_val_fold, train_df=importance_model._train_df)
)
perm_matrix = perm_matrix.reindex(columns=importance_model._feature_columns, fill_value=0.0)
perm_y = winning_transform_obj.forward(perm_val_fold).to_numpy()

is_masked = perm_val_fold["TWS_t"].isna().to_numpy()
regime_importances = {}
for regime_name, mask in [("masked", is_masked), ("observed", ~is_masked)]:
    if mask.sum() < 50:
        print(f"[{regime_name}] fewer than 50 rows in this fold's regime subset -- skipping")
        continue
    result = permutation_importance(
        importance_model._model, perm_matrix.loc[mask], perm_y[mask],
        n_repeats=5, random_state=RANDOM_SEED, scoring="neg_root_mean_squared_error", n_jobs=-1,
    )
    regime_importances[regime_name] = pd.Series(
        result.importances_mean, index=perm_matrix.columns
    ).sort_values(ascending=False)
    print(f"\n[{regime_name}] top 15 permutation-importance features:")
    print(regime_importances[regime_name].head(15).to_string())
print(f"\n({time.time()-t0:.1f}s)")


In [ ]:
# A-008/A-010 sanity check: do ACF-lag / months_since_observation features
# actually dominate specifically in the masked regime, as the architecture predicts?
if "masked" in regime_importances:
    masked_top = regime_importances["masked"].head(15).index.tolist()
    acf_or_staleness_hits = [
        c for c in masked_top
        if "acf" in c.lower() or "months_since_observation" in c.lower() or "blackout_streak" in c.lower()
    ]
    print(f"A-008/A-010 check -- ACF-lag / staleness features in the masked regime's top 15: "
          f"{len(acf_or_staleness_hits)}/15")
    print(acf_or_staleness_hits)


In [ ]:
if importance_report.degradation_slope is not None and not importance_report.degradation_slope.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    for q, color in zip(ACF_QUARTILE_ORDER, ["#d7191c", "#fdae61", "#abd9e9", "#2c7bb6"]):
        sub = importance_report.degradation_slope[
            importance_report.degradation_slope["acf_quartile"] == q
        ].sort_values("k")
        if sub.empty:
            continue
        ax.plot(sub["k"], sub["empirical_rmse"], marker="o", color=color, label=f"{q} (empirical)")
        ax.plot(sub["k"], sub["theoretical_rmse"], color=color, linestyle="--", alpha=0.5)
    ax.set_xlabel("Months since last real observation (k)")
    ax.set_ylabel("RMSE")
    ax.set_title("Degradation slope (Tier 2) -- full assembled feature matrix")
    ax.legend(fontsize=8)
    savefig(fig, "05_degradation_slope_full_matrix.png")
    plt.show()
else:
    print("No degradation slope available for this candidate.")


In [ ]:
decision = promote(importance_report, baseline_report=None)
print(f"Ladder-only promotion check: promoted={decision.promoted}  rung={decision.rung} -- {decision.reason}")

logged = log_candidate(
    importance_report, decision=decision, model_name=f"LightGBMFeaturePredictor ({winning_transform})",
    notes=(
        "notebooks/05_state_features.ipynb, Project Phase 4 step 4.9 -- full assembled feature matrix "
        "(steps 4.1-4.6), feature-importance-by-regime proof run. Not a promotion decision (Project Phase 5 "
        "owns that against the full champion ladder) -- this is Phase 4's own proof that the features built "
        "are not actively harmful and behave the way the architecture predicts."
    ),
)
print(f"Logged as {logged.experiment_id} (MLflow run {logged.mlflow_run_id})")


## 8. Closing synthesis

In [ ]:
print("PROJECT PHASE 4 -- STATE-RECONSTRUCTION FEATURES -- CLOSING SYNTHESIS")
print("="*78)
print()
print("1. LEAKAGE: future_row_shuffle_test on the fully assembled pipeline "
      f"{'PASSED' if leakage_ok else 'FAILED'} -- the composed whole, not just each Transformer in isolation "
      "(step 4.8 covers the units).")
print()
print("2. TARGET TRANSFORMATION (step 4.7):")
print(target_transform_summary.to_string(index=False))
print(f"   Winner: {winning_transform!r}"
      + ("" if winning_transform == "level" else " -- default changed, ADR required (see section 5)."))
print()
print("3. A-014 DIRECT CONFIRMATION:")
print(f"   Naive climatology (Phase 3 reference 1.0796, rerun here = {climatology_report.tier2.overall_rmse:.4f})")
print(f"   Shrinkage signature mean = {signature_report.tier2.overall_rmse:.4f}")
print(f"   Beats naive climatology: {signature_report.tier2.overall_rmse < 1.0796}   "
      f"Beats global mean (0.8740): {signature_report.tier2.overall_rmse < 0.8740}")
print()
print("4. FEATURE IMPORTANCE (full assembled matrix, decomposed by regime):")
print(f"   Tier 2 overall RMSE = {importance_report.tier2.overall_rmse:.4f}  "
      f"(Baseline D floor = {BASELINE_D}, beats floor = {importance_report.tier2.overall_rmse < BASELINE_D})")
print(f"   Top 5 gain-importance features: {gain_importance.head(5).index.tolist()}")
if "masked" in regime_importances:
    print(f"   Top 5 masked-regime permutation-importance features: "
          f"{regime_importances['masked'].head(5).index.tolist()}")
if "observed" in regime_importances:
    print(f"   Top 5 observed-regime permutation-importance features: "
          f"{regime_importances['observed'].head(5).index.tolist()}")
print()
print("Not a promotion decision -- Project Phase 5 (core gradient-boosted forecasting) owns that, against "
      "the full champion ladder. This run is Phase 4's own proof that the feature layer it built works, is "
      "leakage-safe end-to-end, and behaves the way docs/ARCHITECTURE.md predicted it should.")
